<a href="https://colab.research.google.com/github/Kalrfou/Special_Topics2/blob/main/Ollama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Install Ollama in Colab

In [ ]:
!apt-get update -q
!apt-get install -y zstd -q

In [12]:
!pip install pypdf -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 11.1 MB/s eta 0:00:00


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

2. Start Ollama server

In [3]:
import subprocess, time

process = subprocess.Popen(["ollama", "serve"])
time.sleep(5)

3. Download Llama 3.2

In [4]:
!ollama pull llama3.2

4. Install LangChain packages

In [ ]:
!pip install langchain langchain-ollama langchain-chroma langchain-huggingface gradio sentence-transformers -q
!pip install -q langchain-community  chromadb sentence-transformers transformers accelerate gradio markdown

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2")
response = llm.invoke("Hello")
print(response.content)


### 5- *Prepare* the Files and ChromaDB on Google Drive.
Load the documents from Google Drive and create the Chroma database only once. You can perform this step a single time and then commit/save the generated database.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

import os

folder_path1 = "/content/drive/MyDrive/ttu_ar_md"
def fix_arabic_mojibake(text):
    try:
        return text.encode("latin1").decode("utf-8")
    except:
        return text
CHROMA_PATH = "/content/drive/MyDrive/chroma"

In [ ]:
#folder_path = "/content/ttu"
import os
import shutil
import torch
#CHROMA_PATH = "/content/chroma_arabic_db_new"
COLLECTION_NAME = "arabic_registration_office_rag_new"

if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)

os.makedirs(CHROMA_PATH, exist_ok=True)

print("Using Chroma path:", CHROMA_PATH)

In [17]:
from pathlib import Path
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, convert_to_messages
from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [ ]:
from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader,
    PyPDFLoader
)

import os

docs = []

folder_path = folder_path1

# Load Markdown files
md_loader = DirectoryLoader(
    folder_path,
    glob="**/*.md",
    loader_cls=fix_arabic_mojibake(TextLoader),
    loader_kwargs={"encoding": "utf-8"}
)

docs.extend(md_loader.load())

# Load PDF files
pdf_loader = DirectoryLoader(
    folder_path,
    glob="**/*.pdf",
    loader_cls=PyPDFLoader
)

docs.extend(pdf_loader.load())

print(f"Loaded {len(docs)} documents")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=150
)

chunks = splitter.split_documents(docs)

print("Number of chunks:", len(chunks))

In [ ]:
'''
embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)
'''
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

In [ ]:
if os.path.exists(CHROMA_PATH):
    Chroma(persist_directory=CHROMA_PATH, embedding_function=embedding_model).delete_collection()

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_PATH
)
print("Vectors in Chroma:", vector_db._collection.count())
retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print("ChromaDB created successfully.")

In [ ]:
retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

question = "ما هي عمادة شؤون الطلبة؟"

retrieved_docs = retriever.invoke(question)

for i, doc in enumerate(retrieved_docs, 1):
    print("=" * 50)
    print("Document:", i)
    print("Source:", doc.metadata.get("source", "Unknown"))
    print(doc.page_content[:700])

In [ ]:
collection = vector_db._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

6. RAG answer function with Llama 3.2

In [46]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, convert_to_messages

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

SYSTEM_PROMPT = """
أنت مساعد ذكي باللغة العربية تابع لجامعة الطفيلة التقنية.
مهمتك هي الإجابة عن أسئلة المستخدم اعتمادًا فقط على السياق المسترجع من ملفات موقع الجامعة.

التعليمات:
- أجب باللغة العربية.
- استخدم المعلومات الموجودة في السياق فقط.
- إذا لم تجد الإجابة في السياق، قل: لا أملك معلومات كافية من الملفات المتاحة للإجابة عن هذا السؤال.
- لا تخترع معلومات غير موجودة في السياق.
- اجعل الإجابة واضحة ومنظمة.

السياق:
{context}
"""

def fetch_context(question):
    return retriever.invoke(question)


def combined_question(question, history=None):
    if history is None:
        history = []

    previous_questions = "\n".join(
        msg["content"] for msg in history if msg["role"] == "user"
    )

    return previous_questions + "\n" + question


def answer_question(question, history=None):
    if history is None:
        history = []

    search_query = combined_question(question, history)

    docs = fetch_context(search_query)

    context = "\n\n".join(
        f"Source: {doc.metadata.get('source', 'Unknown')}\n{doc.page_content}"
        for doc in docs
    )

    system_prompt = SYSTEM_PROMPT.format(context=context)

    messages = [SystemMessage(content=system_prompt)]
    messages.extend(convert_to_messages(history))
    messages.append(HumanMessage(content=question))

    response = llm.invoke(messages)

    return response.content, docs

In [ ]:
answer, context_docs = answer_question("ما هي عمادة شؤون الطلبة؟")

print(answer)

print("\nSources:")
for doc in context_docs:
    print(doc.metadata.get("source", "Unknown"))

In [ ]:
import gradio as gr

def format_context(context):
    result = """
    <div style='direction: rtl; text-align: right; font-family: Arial;'>
    <h2 style='color: #ff7800;'>السياق المسترجع من Chroma</h2>
    """

    for i, doc in enumerate(context, 1):
        source = doc.metadata.get("source", "Unknown source")
        text = doc.page_content.replace("\n", "<br>")

        result += f"""
        <div style='border: 1px solid #ddd; padding: 10px; margin-bottom: 10px; border-radius: 8px;'>
            <b style='color: #ff7800;'>المصدر {i}:</b><br>
            <span>{source}</span>
            <hr>
            <p>{text}</p>
        </div>
        """

    result += "</div>"
    return result


def chat(history):
    last_message = history[-1]["content"]
    prior = history[:-1]

    answer, context = answer_question(last_message, prior)

    history.append({
        "role": "assistant",
        "content": answer
    })

    return history, format_context(context)


def put_message_in_chatbot(message, history):
    if history is None:
        history = []

    return "", history + [{"role": "user", "content": message}]


theme = gr.themes.Soft(font=["Arial", "system-ui", "sans-serif"])

with gr.Blocks(title="TTU Arabic RAG Assistant", theme=theme) as ui:
    gr.Markdown(
        """
        <div style='direction: rtl; text-align: right;'>
        <h1>🤖 مساعد جامعة الطفيلة التقنية</h1>
        <p>اسأل عن المعلومات المستخرجة من ملفات موقع الجامعة.</p>
        </div>
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            chatbot = gr.Chatbot(
                label="المحادثة",
                height=600,
                type="messages",
                show_copy_button=True,
                rtl=True
            )

            message = gr.Textbox(
                label="سؤالك",
                placeholder="اكتب سؤالك هنا...",
                show_label=False,
                rtl=True
            )

        with gr.Column(scale=1):
            context_box = gr.HTML(
                label="السياق المسترجع",
                value="<p style='direction: rtl; text-align: right;'>سيظهر السياق المسترجع هنا.</p>"
            )

    message.submit(
        put_message_in_chatbot,
        inputs=[message, chatbot],
        outputs=[message, chatbot]
    ).then(
        chat,
        inputs=chatbot,
        outputs=[chatbot, context_box]
    )

ui.launch(share=True)